# Deep Dive into LangChain
## LLMs, Prompt Templates, Caching, Streaming, Chains

This notebook uses the **langchain-classic** library.

Make sure to import from **langchain_classic** rather than **langchain**.

In [2]:
pip install -r ./requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


Download [requirements.txt](https://drive.google.com/file/d/1oek3_JLI97qy4YZDc2UmmaKXfH7n9FJw/view?usp=sharing)

In [1]:
pip show langchain-classic


Name: langchain-classic
Version: 1.0.2
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/dda/projects/python/Claude-Code-Projects/.venv/lib/python3.13/site-packages
Requires: langchain-core, langchain-text-splitters, langsmith, pydantic, pyyaml, requests, sqlalchemy
Required-by: langchain-community
Note: you may need to restart the kernel to use updated packages.


### Python-dotenv

In [5]:
import os
from dotenv import load_dotenv, find_dotenv

# loading the API Keys from .env
load_dotenv(find_dotenv(), override=True)

# os.environ.get('OPENAI_API_KEY')

True

## OpenAI Chat Models

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()  

# invoking the llm (running the prompt)
output = llm.invoke('Explain quantum mechanics in one sentence.', model='gpt-5.1', temperature=0.1)
print(output.content)

Quantum mechanics is the theory that describes how matter and energy behave at very small scales, where particles act like both waves and particles and outcomes are fundamentally probabilistic rather than strictly determined.


In [6]:
help(ChatOpenAI)  # see the llm constructor arguments with its defaults

Help on class ChatOpenAI in module langchain_openai.chat_models.base:

class ChatOpenAI(BaseChatOpenAI)
 |  ChatOpenAI(
 |      *args: Any,
 |      name: str | None = None,
 |      cache: langchain_core.caches.BaseCache | bool | None = None,
 |      verbose: bool = <factory>,
 |      callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None,
 |      tags: list[str] | None = None,
 |      metadata: dict[str, typing.Any] | None = None,
 |      custom_get_token_ids: collections.abc.Callable[[str], list[int]] | None = None,
 |      rate_limiter: langchain_core.rate_limiters.BaseRateLimiter | None = None,
 |      disable_streaming: Union[bool, Literal['tool_calling']] = False,
 |      output_version: str | None = <factory>,
 |      profile: langchain_core.language_models.model_profile.ModelProfile | None = None,
 |      client: Any = None,
 |      async_client: Any = None,
 |      root_client: Any = None,
 |      roo

In [4]:
# using Chat Completions API Messages: System, Assistant and Human
from langchain_classic.schema import(
    SystemMessage, 
    AIMessage,
    HumanMessage
)
messages = [
    SystemMessage(content='You are a physicist and respond only in German.'),
    HumanMessage(content='Explain quantum mechanics in one sentence.')
]

output = llm.invoke(messages)
print(output.content)

Quantenmechanik beschreibt das Verhalten von Teilchen auf mikroskopischer Ebene durch Wellenfunktionen und Wahrscheinlichkeiten.


## Caching LLM Responses

### 1. In-Memory Cache

In [5]:
from langchain_classic.globals import set_llm_cache
from langchain_openai import OpenAI
llm = OpenAI(model_name='gpt-4o-mini')

In [6]:
from langchain_classic.cache import InMemoryCache
set_llm_cache(InMemoryCache())

In [7]:
%%time
prompt = 'Tell a me a joke that a toddler can understand.'
llm.invoke(prompt)

CPU times: user 18.7 ms, sys: 985 μs, total: 19.7 ms
Wall time: 7.54 s


' Why did the teddy bear say no to dessert? Because it was already stuffed! �����🤣\n\nWhat did the big flower say to the little flower? "Hey, bud!" ������\n\nWhy did the cookie go to the doctor? Because it felt crummy! ����😂\n\nWhat do you call a dinosaur that is sleeping? A dino-snore! �������\n\nWhy do seagulls fly over the ocean? Because if they flew over the bay, they’d be bagels! ������️😂\n\nWhat do you call cheese that isn\'t yours? Nacho cheese! ���🤣��\n\nWhy did the banana go to the doctor? Because it wasn\'t peeling well! ����😂\n\nWhy did the cow go to outer space? To see the moooon! �������\n\nWhy did the chicken cross the playground? To get to the other slide! �����😂\n\nWhat do you call a bear with no teeth? A gummy bear! �������\n\nWhy did the duck go to the mall? Because he wanted to buy some quackers! �����'

In [8]:
%%time
llm.invoke(prompt)

CPU times: user 206 μs, sys: 0 ns, total: 206 μs
Wall time: 210 μs


' Why did the teddy bear say no to dessert? Because it was already stuffed! �����🤣\n\nWhat did the big flower say to the little flower? "Hey, bud!" ������\n\nWhy did the cookie go to the doctor? Because it felt crummy! ����😂\n\nWhat do you call a dinosaur that is sleeping? A dino-snore! �������\n\nWhy do seagulls fly over the ocean? Because if they flew over the bay, they’d be bagels! ������️😂\n\nWhat do you call cheese that isn\'t yours? Nacho cheese! ���🤣��\n\nWhy did the banana go to the doctor? Because it wasn\'t peeling well! ����😂\n\nWhy did the cow go to outer space? To see the moooon! �������\n\nWhy did the chicken cross the playground? To get to the other slide! �����😂\n\nWhat do you call a bear with no teeth? A gummy bear! �������\n\nWhy did the duck go to the mall? Because he wanted to buy some quackers! �����'

### 2. SQLite Caching

In [9]:
from langchain_classic.cache import SQLiteCache
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

In [10]:
%%time
# First request (not in cache, takes longer)
llm.invoke("Tell me a joke")

CPU times: user 20.8 ms, sys: 3.92 ms, total: 24.8 ms
Wall time: 4.84 s


" about a cat and a dog.**  \n**Why did the cat sit on the dog’s head?**  \n**Because it wanted to be the “purr-sident” of the pack!**  \n\n**What's a cat's favorite color?**  \n**Purr-ple!**  \n\n**How do cats end a fight?**  \n**They hiss and make up!**  \n\n**What do you call a dog that can do magic?**  \n**A labracadabrador!**  \n\n**Why did the cat and dog start a band?**  \n**Because they were great at “pawsing” for the right notes!**  \n\n**What do you get when you cross a cat and a dog?**  \n**A fur-tunate friendship!**  \n\n**Why did the dog sit in the shade?**  \n**Because he didn’t want to become a hot dog!**  \n\n**What did the cat say to the dog on their road trip?**  \n**“Let’s take the meow-torway!”**  \n\n**What do you call a cat who loves to swim?**  \n**A “purr-l swimmer”!**  \n\nI hope these brought a smile to your face! If you'd like to hear more jokes"

In [11]:
%%time
# Second request (cached, faster)
llm.invoke("Tell me a joke")

CPU times: user 362 μs, sys: 1.02 ms, total: 1.38 ms
Wall time: 1.29 ms


" about a cat and a dog.**  \n**Why did the cat sit on the dog’s head?**  \n**Because it wanted to be the “purr-sident” of the pack!**  \n\n**What's a cat's favorite color?**  \n**Purr-ple!**  \n\n**How do cats end a fight?**  \n**They hiss and make up!**  \n\n**What do you call a dog that can do magic?**  \n**A labracadabrador!**  \n\n**Why did the cat and dog start a band?**  \n**Because they were great at “pawsing” for the right notes!**  \n\n**What do you get when you cross a cat and a dog?**  \n**A fur-tunate friendship!**  \n\n**Why did the dog sit in the shade?**  \n**Because he didn’t want to become a hot dog!**  \n\n**What did the cat say to the dog on their road trip?**  \n**“Let’s take the meow-torway!”**  \n\n**What do you call a cat who loves to swim?**  \n**A “purr-l swimmer”!**  \n\nI hope these brought a smile to your face! If you'd like to hear more jokes"

## LLM Streaming

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
prompt = 'Write a rock song about the Moon and a Raven.'
print(llm.invoke(prompt).content)

Verse 1:
Underneath the moon's silver light
A raven takes flight, through the night
Its black wings shining against the sky
As it calls out its haunting cry

Chorus:
Moon and raven, dance in the night
Their souls intertwined, in the pale moonlight
A mysterious bond, never to sever
Rocking out with the moon forever

Verse 2:
The raven watches from its perch
As the moon illuminates the earth
Its eyes gleaming with ancient wisdom
As it soars through the midnight kingdom

Chorus:
Moon and raven, dance in the night
Their souls intertwined, in the pale moonlight
A mysterious bond, never to sever
Rocking out with the moon forever

Bridge:
The night is alive with their song
A symphony of darkness, echoing strong
The moon and raven, a perfect pair
In the night sky, they both declare

Chorus:
Moon and raven, dance in the night
Their souls intertwined, in the pale moonlight
A mysterious bond, never to sever
Rocking out with the moon forever

Outro:
So next time you gaze up at the moon
Think of th

In [13]:
for chunk in llm.stream(prompt):
    print(chunk.content, end='', flush=True)

Verse 1:
Beneath the veil of midnight's sky
The moon shines bright, a silvery eye
A raven calls from yonder tree
Singing songs of mystery

Chorus:
Oh moon, oh raven
Dance in the night
Soar high above
In the pale moonlight

Verse 2:
The raven's wings, black as coal
Against the moon, a haunting patrol
Through the darkness, they take flight
Guided by the lunar light

Chorus:
Oh moon, oh raven
Dance in the night
Soar high above
In the pale moonlight

Bridge:
In the shadows, secrets lie
Beneath the moon, the raven cries
A symbiotic bond they share
In the stillness of the midnight air

Chorus:
Oh moon, oh raven
Dance in the night
Soar high above
In the pale moonlight

Outro:
As the night fades into dawn
The moon and raven carry on
A timeless dance, forevermore
In the celestial encore

## PromptTemplates

In [14]:
from langchain_classic.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Define a template for the prompt
template = '''You are an experienced virologist.
Write a few sentences about the following virus "{virus}" in {language}.'''

# Create a PromptTemplate object from the template
prompt_template = PromptTemplate.from_template(template=template)

# Fill in the variable: virus and language
prompt = prompt_template.format(virus='hiv', language='german')
prompt  # Returns the generated prompt


'You are an experienced virologist.\nWrite a few sentences about the following virus "hiv" in german.'

In [15]:
llm = ChatOpenAI(model_name='gpt-5-nano', temperature=0)
output = llm.invoke(prompt)
print(output.content)

Als Virologe kann ich sagen: Das humane Immundefizienz-Virus (HIV) ist ein Lentivirus der Retroviridae-Familie, das das Immunsystem schädigt. Es befällt primär CD4-positive T-Helferzellen und nutzt gp120/gp41 in Verbindung mit den Co-Rezeptoren CCR5 oder CXCR4, um in die Zelle einzudringen. Durch Integration seines proviralen Genoms ins Wirtsgenom und eine hohe Mutationsrate führt eine unbehandelte Infektion zu fortschreitendem Immunverlust und AIDS. Weltweit dominiert HIV-1; HIV-2 kommt vor allem in Westafrika vor und verläuft tendenziell milder. Die antiretrovirale Therapie (ART) in Kombination unterdrückt die Virusreplikation, stabilisiert das Immunprofil und erhöht die Lebensdauer; Präventionsmaßnahmen umfassen Safer-Sex-Praktiken, PrEP/PEP sowie sichere Blutprodukte.


## ChatPromptTemplates

In [16]:
from langchain_classic.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

# Create a chat template with system and human messages
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content='You respond only in the JSON format.'),
        HumanMessagePromptTemplate.from_template('Top {n} countries in {area} by population.')
    ]
)

# Fill in the specific values for n and area
messages = chat_template.format_messages(n='5', area='World')
print(messages)  # Outputs the formatted chat messages


[SystemMessage(content='You respond only in the JSON format.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Top 5 countries in World by population.', additional_kwargs={}, response_metadata={})]


In [17]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()
output = llm.invoke(messages)
print(output.content)

{
    "1": "China",
    "2": "India",
    "3": "United States",
    "4": "Indonesia",
    "5": "Pakistan"
}


## Simple Chains

In [7]:
from langchain_openai import ChatOpenAI
from langchain_classic import PromptTemplate
from langchain_classic.chains import LLMChain

llm = ChatOpenAI()
template = '''You are an experience virologist.
Write a few sentences about the following virus "{virus}" in {language}.'''
prompt_template = PromptTemplate.from_template(template=template)

# deprecated - see below how to update to the new chain syntax
chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    verbose=True
)
output = chain.invoke({'virus': 'HSV', 'language': 'Spanish'})
print(output)



> Entering new LLMChain chain...
Prompt after formatting:
You are an experience virologist.
Write a few sentences about the following virus "HSV" in Spanish.

> Finished chain.
{'virus': 'HSV', 'language': 'Spanish', 'text': 'El virus del herpes simple, o HSV por sus siglas en inglés, es un virus de la familia Herpesviridae que afecta a los seres humanos. Existen dos tipos de HSV: el HSV-1, que generalmente causa lesiones en la zona bucal, y el HSV-2, que suele provocar herpes genital. Ambos tipos de herpes pueden transmitirse a través del contacto directo con una persona infectada y pueden causar síntomas como ampollas, úlceras y dolor. El HSV es un virus común y puede permanecer en el cuerpo de forma latente, causando brotes recurrentes.'}


In [8]:
# if the code in the cell above gives an LLMChain deprecation error replace:
# chain = LLMChain(
#     llm=llm,
#     prompt=prompt_template,
#     verbose=True
# )

# with: 

from langchain_core.output_parsers import StrOutputParser

chain = prompt_template | llm | StrOutputParser()
output = chain.invoke({'virus': 'HSV', 'language': 'Spanish'})
print(output)

El virus del herpes simple (HSV) es un virus de la familia de los herpes que afecta a los seres humanos. Hay dos tipos de HSV: el HSV-1, que generalmente causa herpes oral, y el HSV-2, que se asocia con herpes genital. Ambos tipos de herpes pueden causar lesiones en la piel y en las membranas mucosas. El HSV es altamente contagioso y se transmite principalmente a través del contacto directo con una persona infectada.


In [9]:
template = 'What is the capital of {country}?. List the top 3 places to visit in that city. Use bullet points'
prompt_template = PromptTemplate.from_template(template=template)

# Initialize an LLMChain with the ChatOpenAI model and the prompt template
chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    verbose=True
)

country = input('Enter Country: ')

# Invoke the chain with specific virus and language values
output = chain.invoke(country)
print(output['text'])

Enter Country:  Romania




> Entering new LLMChain chain...
Prompt after formatting:
What is the capital of Romania?. List the top 3 places to visit in that city. Use bullet points

> Finished chain.
The capital of Romania is Bucharest.

Top 3 places to visit in Bucharest:
- Palace of the Parliament
- Old Town (Lipscani)
- Herastrau Park


In [10]:
# Updated to use the new LCEL (LangChain Expression Language) syntax
# Instead of LLMChain, we use the pipe operator (|) to chain components
from langchain_core.output_parsers import StrOutputParser

chain = prompt_template | llm | StrOutputParser()

country = input('Enter Country: ')

# Invoke the chain with a dictionary of input variables
output = chain.invoke({'country': country})
print(output)

Enter Country:  Spain


The capital of Spain is Madrid. 

Some of the top places to visit in Madrid include:

- The Royal Palace
- Prado Museum
- Retiro Park


## Sequential Chains

In [11]:
from langchain_openai import ChatOpenAI
from langchain_classic import PromptTemplate
from langchain_classic.chains import LLMChain, SimpleSequentialChain

# Initialize the first ChatOpenAI model with specific temperature
llm1 = ChatOpenAI(model_name='gpt-5.1', temperature=0.5)

# Define the first prompt template
prompt_template1 = PromptTemplate.from_template(
    template='You are an experienced scientist and Python programmer. Write a function that implements the concept of {concept}.'
)

# DEPRECATED - see below how to update to the new chain syntax

# Create an LLMChain using the first model and the prompt template
chain1 = LLMChain(llm=llm1, prompt=prompt_template1)

# Initialize the second ChatOpenAI model with specific temperature
llm2 = ChatOpenAI(model_name='gpt-5.1', temperature=1.2)

# Define the second prompt template
prompt_template2 = PromptTemplate.from_template(
    template='Given the Python function {function}, describe it as detailed as possible.'
)
# Create another LLMChain using the second model and the prompt template
chain2 = LLMChain(llm=llm2, prompt=prompt_template2)

# Combine both chains into a SimpleSequentialChain
overall_chain = SimpleSequentialChain(chains=[chain1, chain2], verbose=True)

# Invoke the overall chain with the concept "linear regression"
output = overall_chain.invoke('linear regression')




> Entering new SimpleSequentialChain chain...
Here’s a clear, from-scratch implementation of simple linear regression (one feature) using the ordinary least squares (OLS) formula.

```python
import numpy as np

def linear_regression(x, y):
    """
    Fit a simple linear regression model y = a*x + b using least squares.
    
    Parameters
    ----------
    x : array-like, shape (n,)
        1D array of input features.
    y : array-like, shape (n,)
        1D array of target values.
    
    Returns
    -------
    a : float
        Slope of the best-fit line.
    b : float
        Intercept of the best-fit line.
    """
    # Convert to numpy arrays
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    
    if x.shape != y.shape:
        raise ValueError("x and y must have the same shape.")
    if x.ndim != 1:
        raise ValueError("Only 1D input supported for this simple implementation.")
    if x.size < 2:
        raise ValueError("At least two data points

In [12]:
print(output['output'])

This function implements **simple linear regression** (one predictor variable) from first principles using the **ordinary least squares (OLS)** closed‑form solution.

I’ll walk through:

1. What problem it solves  
2. The math behind it  
3. How each part of the code corresponds to the math  
4. Important edge cases and error checks  
5. Example interpretation of the output  
6. Brief note on extending to multivariate regression  

---

## 1. Problem it solves

We have data points:

- Inputs (feature): \( x_1, x_2, ..., x_n \)  
- Outputs (target): \( y_1, y_2, ..., y_n \)

We want to fit a **straight line**:

\[
\hat{y} = a x + b
\]

where:

- \( a \) is the **slope**
- \( b \) is the **intercept**
- \( \hat{y} \) is the predicted value of \( y \) for a given \( x \)

The goal is to choose \( a \) and \( b \) such that the line best fits the data in the sense of **least squares**, i.e., it minimizes the sum of squared vertical errors (residuals):

\[
\text{minimize over } a, b: \quad 

In [20]:
# Updated to use the new LCEL (LangChain Expression Language) syntax
# Instead of LLMChain and SimpleSequentialChain, we use the pipe operator (|) to chain components
from langchain_openai import ChatOpenAI
from langchain_classic import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm1 = ChatOpenAI(model_name='gpt-5-nano', temperature=0)
llm2 = ChatOpenAI(model_name='gpt-5.1', temperature=1.2)

prompt_template1 = PromptTemplate.from_template(
    template='You are an experienced scientist and Python programmer. Write a function that implements the concept of {concept}.'
)
prompt_template2 = PromptTemplate.from_template(
    template='Given the Python function {function}, describe it as detailed as possible.'
)

# Chain 1: concept -> Python function (as string)
chain1 = prompt_template1 | llm1 | StrOutputParser()

# Chain 2: function string -> detailed description
chain2 = prompt_template2 | llm2 | StrOutputParser()

# Connect chains: output of chain1 becomes {function} input for chain2
overall_chain = chain1 | (lambda text: {'function': text}) | chain2

# Invoke the overall chain with the concept "linear regression"
output = overall_chain.invoke({'concept': 'linear regression'})
print(output)

This implementation is a compact, from-scratch version of linear regression in Python using NumPy. It supports:

- Multiple input features
- Optional intercept term
- Optional L2 (ridge) regularization
- Two solvers: closed-form (normal equations) and gradient descent
- A separate prediction function

Below is a detailed walkthrough of what it does and how it works.

---

## 1. Function: `fit_linear_regression`

Signature:

```python
def fit_linear_regression(
    X, y,
    intercept=True,
    reg_lambda=0.0,
    solver='auto',
    max_iter=1000,
    learning_rate=0.01,
    tol=1e-9,
    verbose=False
)
```

### Inputs

- `X`:  
  - Shape `(n_samples, n_features)` or `(n_samples,)`  
  - Features / predictors matrix.
- `y`:  
  - Shape `(n_samples,)` or `(n_samples, 1)`  
  - Target values.

- `intercept` (bool):  
  Whether to include a bias/intercept term \( b \) in the model. If `True` (default), the model is:
  \[
  \hat{y} = b + X\beta
  \]
  If `False`:
  \[
  \hat{y} = X\beta
  

In [ ]:
# pip install -q --upgrade langchain_experimental pydantic

Note: you may need to restart the kernel to use updated packages.


In [14]:
from langchain_experimental.utilities import PythonREPL

python_repl = PythonREPL()
python_repl.run('print([n for n in range(1, 100) if n % 13 == 0])')


Python REPL can execute arbitrary code. Use with caution.


'[13, 26, 39, 52, 65, 78, 91]\n'

In [15]:
# Updated code 

from langchain_openai import ChatOpenAI
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
tools = [PythonREPLTool()]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python coder. Use the Python REPL for any math/code. Return only the final answer."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({
    "input": "Calculate the square root of the factorial of 12 and display it with 4 decimal points"
})




> Entering new AgentExecutor chain...

Invoking: `Python_REPL` with `{'query': 'print((479001600)**0.5)'}`


21886.105181141756
21886.1052

> Finished chain.


{'input': 'Calculate the square root of the factorial of 12 and display it with 4 decimal points',
 'output': '21886.1052'}

In [16]:
response = agent_executor.invoke({
    'input': 'What is the answer to 5.1 ** 7.3?'
})



> Entering new AgentExecutor chain...

Invoking: `Python_REPL` with `{'query': 'print(5.1 ** 7.3)'}`


146306.05007233328
146306.05007233328

> Finished chain.


In [17]:
response

{'input': 'What is the answer to 5.1 ** 7.3?', 'output': '146306.05007233328'}

In [18]:
print(response['input'])

What is the answer to 5.1 ** 7.3?


In [19]:
print(response['output'])


146306.05007233328
